# 09. Сравнение original и corrected_v1

Сводит метрики запусков, распределения классов и изменения gold-разметки. Validation показывает эффект для внутреннего эксперимента. Test original и corrected выводятся раздельно, поскольку их gold-разметка различается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, pandas as pd
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
ORIGINAL_RUN = PROJECT_DIR / 'results/ner_baseline/ner_baseline_v1'
CORRECTED_RUNS = {
    42: PROJECT_DIR / 'results/ner_baseline_corrected_v1/seed_42',
    17: PROJECT_DIR / 'results/ner_baseline_corrected_v1/seed_17',
    73: PROJECT_DIR / 'results/ner_baseline_corrected_v1/seed_73',
}
def read_json(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))

In [ ]:
rows = []
original_train = read_json(ORIGINAL_RUN / 'train_metrics.json')
original_test = read_json(ORIGINAL_RUN / 'test_metrics.json')
rows.append({'dataset':'original', 'seed':42, 'validation_micro_f1':original_train['best_validation_f1'], 'test_micro_f1':original_test['micro_f1'], 'test_gold':'official_original'})
for seed, run_dir in CORRECTED_RUNS.items():
    train = read_json(run_dir / 'train_metrics.json')
    test = read_json(run_dir / 'test_metrics.json')
    official_test = read_json(run_dir / 'official_original_test_metrics.json')
    rows.append({'dataset':'corrected_v1', 'seed':seed, 'validation_micro_f1':train['best_validation_f1'], 'test_micro_f1':test['micro_f1'], 'official_original_test_micro_f1':official_test['micro_f1'], 'test_gold':'internal_corrected'})
metrics = pd.DataFrame(rows)
metrics.loc[metrics.dataset == 'original', 'official_original_test_micro_f1'] = metrics.loc[metrics.dataset == 'original', 'test_micro_f1']
display(metrics)
display(metrics.groupby('dataset')[['validation_micro_f1','test_micro_f1','official_original_test_micro_f1']].agg(['mean','std']))

In [ ]:
original_manifest = pd.read_csv(PROJECT_DIR / 'rurebus_data/processed/manifest.csv', encoding='utf-8-sig')
corrected_manifest = pd.read_csv(PROJECT_DIR / 'rurebus_data/versions/corrected_v1/manifest.csv', encoding='utf-8-sig')
dataset_diff = original_manifest[['document_id','split','ann_sha256']].merge(corrected_manifest[['document_id','ann_sha256']], on='document_id', suffixes=('_original','_corrected'))
print('Документов с изменившейся ANN-разметкой:', (dataset_diff.ann_sha256_original != dataset_diff.ann_sha256_corrected).sum())
display(dataset_diff.loc[dataset_diff.ann_sha256_original != dataset_diff.ann_sha256_corrected].head(20))
print('Не интерпретируйте разность test F1 как чистое улучшение модели: test gold изменился.')